# Step-by-step demo of NEDAS using the vort2d model

## Load NEDAS and dependencies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from NEDAS import get_scheme
from NEDAS.config import Config

## Initialize configuration and main scheme

In [2]:
config = Config(config_file="vort2d/config.yml")

In [3]:
scheme = get_scheme(config)

model = scheme.c.models['vort2d']
dataset = scheme.c.datasets['vort2d']

In [4]:
# jupyter notebook env doesn't manage rng well, patching the funcs to make sure seed is refreshed.
original_generate_ic = model.generate_initial_condition
def patched_generate_ic(*args, **kwargs):
    np.random.seed()
    return original_generate_ic(*args, **kwargs)
model.generate_initial_condition = patched_generate_ic

In [5]:
grid = scheme.c.grid

In [6]:
from NEDAS.utils.spatial_operation import gradx, grady
def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta


## Prepare truth

check nc files in vort2d/truth

In [12]:
%rm -rf vort2d/work

In [16]:
scheme.c.time = config.time_start

In [15]:
model.loc_sprd = 0
scheme.prepare_truth()

## Prepare initial ensemble and run forecasts

In [19]:
model.loc_sprd = 10000
scheme.prepare_init_ensemble()

├── Generate vort2d init ensemble ─────────────── ✅     0.80s (all 16 jobs done)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         


In [20]:
scheme.c.progress.call_stack = []
scheme.c.progress.call_stack_max_level = 2
scheme.c.progress.push('')
scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')


│   
◈ CYCLING START...
│   
◈ CURRENT CYCLE: 2001-01-01 00:00:00+00:00 -> 2001-01-01 03:00:00+00:00
├── Running preprocess step ───────────────────── ✅     0.82s (all 16 jobs done)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
├── Running ensemble_forecast step ────────────── ✅    21.53s (all 16 jobs done)       

/opt/python/lib/python3.13/site-packages/NEDAS/models/vort2d/util.py:161: RuntimeWarning: overflow encountered in multiply
  f = -fft2(ug*ifft2(1j*ki*zeta) + vg*ifft2(1j*kj*zeta))


├── Running ensemble_forecast step > Run vort2d forecast... ⏳  [━─────────]  18% (1/16 jobs done, 8 running)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

/opt/python/lib/python3.13/site-packages/NEDAS/models/vort2d/util.py:161: RuntimeWarning: overflow encountered in add
  f = -fft2(ug*ifft2(1j*ki*zeta) + vg*ifft2(1j*kj*zeta))
/opt/python/lib/python3.13/site-packages/NEDAS/models/vort2d/util.py:161: RuntimeWarning: invalid value encountered in add
  f = -fft2(ug*ifft2(1j*ki*zeta) + vg*ifft2(1j*kj*zeta))


├── Running ensemble_forecast step ────────────── ✅    19.31s (all 16 jobs done)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
│   
🏁  CYCLING COMPLETE.


collect data for diagnostics

In [21]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(truth)
    scheme.c.time = scheme.c.next_time

In [22]:
scheme.c.time = config.time_start
prior_state = []
for n in range(10):
    path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)
    scheme.c.time = scheme.c.next_time

## Substeps in an analysis cycle

In [12]:
scheme.c.time = config.time_start

## Running the entire scheme

Cycling from time_start to time_end

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []
scheme.c.progress.call_stack_max_level = 2

scheme()


In [ ]:
scheme.c.time = config.time_start
truth_state = []
prior_state = []
post_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(truth)

    path = scheme.c.fs.forecast_dir(scheme.c.prev_time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)

    ens = []
    for m in range(scheme.c.nens):
        if config.run_analysis and scheme.c.time >= config.time_analysis_start and scheme.c.time <= config.time_analysis_end:
            path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
            ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(truth.shape, np.nan))
    post_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [23]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
nt = 10 #len(truth_state)
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove() 
    current_contour_sets.clear()

    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        var = to_vorticity(prior_state[n][m])
        cs_ens = ax.contour(grid.x, grid.y, var, [1e-3], colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    # for m in range(scheme.c.nens):
    #     var = to_vorticity(post_state[n][m])
    #     cs_ens = ax.contour(grid.x, grid.y, var, [1e-3], colors='b', linewidths=0.5)
    #     current_contour_sets.append(cs_ens)

    var = to_vorticity(truth_state[n])
    cs_truth = ax.contour(grid.x, grid.y, var, [1e-3], colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)

    ax.set_title(f"Time: {times[n]}")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## A few things to try to play with
